# AI Summary Writer (Phase 5)

A fourth independent background process, same pattern as `live_scoreboard_writer.ipynb`: 
it wakes up periodically, reads the current scoreboard files, asks an LLM (via Groq — free, 
fast, no OpenAI cost) to write a short business summary, and saves that summary to its own 
file for the dashboard to display.

**Why this runs on its own slow timer (default: every 90 seconds), not every 5 seconds like 
the numbers:** an LLM call takes a few seconds and the data doesn't meaningfully change second 
to second anyway. Calling it as often as the charts refresh would be pointless and would burn 
through Groq's rate limits fast.

**Setup:**
1. Get a free API key at console.groq.com if you don't already have one.
2. Set it as an environment variable before launching Jupyter: `GROQ_API_KEY=gsk_...`
3. Run this notebook continuously, alongside the generator, poller, and scoreboard writer.


In [8]:
!pip install groq -q


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    raise EnvironmentError(
        "Set GROQ_API_KEY as an environment variable (or paste it above as a fallback "
        "string, though an env var is safer). Get a free key at console.groq.com."
    )

client = Groq(api_key=GROQ_API_KEY)
MODEL = "llama-3.3-70b-versatile"  # same model used in the Arbiter project

In [10]:
from pathlib import Path

SCOREBOARD_DIR = Path("dashboard_scoreboard")  # reads from here, same folder the scoreboard writer uses
print("Reading scoreboard from:", SCOREBOARD_DIR.resolve())

Reading scoreboard from: C:\Users\chait\Documents\spark_projects\dashboard_scoreboard


## Read the current scoreboard and build a prompt

In [11]:
import json

def load_json(filename):
    path = SCOREBOARD_DIR / filename
    if not path.exists():
        return None
    try:
        with open(path, "r") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        return None


def build_prompt():
    """Read the latest scoreboard files and turn them into a compact prompt. Returns None if
    there isn't enough data yet to summarize."""
    overall = load_json("overall_stats.json")
    products = load_json("product_revenue.json")
    states = load_json("state_revenue.json")
    fulfillment = load_json("fulfillment_stats.json")

    if not overall or not products:
        return None

    top_products = sorted(products, key=lambda p: p["revenue"], reverse=True)[:5]
    top_products_str = ", ".join(f"{p['product_name']} (₹{p['revenue']:,.0f})" for p in top_products)

    status_str = ", ".join(
        f"{s['status']}: {s['order_count']} orders (₹{s['revenue']:,.0f})"
        for s in overall.get("by_status", [])
    )

    top_states = sorted(states or [], key=lambda s: s["revenue"], reverse=True)[:5]
    states_str = (
        ", ".join(f"{s['state']} (₹{s['revenue']:,.0f})" for s in top_states)
        if top_states else "no data yet"
    )

    if fulfillment and fulfillment.get("avg_fulfillment_minutes") is not None:
        fulfillment_str = f"{fulfillment['avg_fulfillment_minutes']:.1f} minutes across {fulfillment['completed_count']} completed orders"
    else:
        fulfillment_str = "not enough completed orders yet"

    prompt = f"""You are a retail analytics assistant. Based on this live e-commerce snapshot, write a concise 3-4 sentence business summary in plain prose (no bullet points, no headers), under 90 words. Cover: what's driving revenue, any notable pattern in order status, and one operational observation.

Data snapshot:
- Total orders: {overall['total_orders']}, Total revenue: ₹{overall['total_revenue']:,.2f}, Avg order value: ₹{overall['avg_order_value']:,.2f}
- Top products by revenue: {top_products_str}
- Orders by status: {status_str}
- Revenue by state (top 5): {states_str}
- Avg fulfillment time: {fulfillment_str}
"""
    return prompt

## Call Groq and write the summary

In [12]:
import groq
from datetime import datetime

def generate_and_write_summary():
    prompt = build_prompt()
    if prompt is None:
        print(f"[{datetime.now()}] Not enough data yet, skipping this cycle.")
        return

    try:
        response = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model=MODEL,
            max_completion_tokens=200,
            temperature=0.4,
        )
        summary_text = response.choices[0].message.content.strip()
    except groq.RateLimitError:
        print(f"[{datetime.now()}] Rate limited by Groq — will try again next cycle.")
        return
    except groq.APIConnectionError as e:
        print(f"[{datetime.now()}] Could not reach Groq: {e}")
        return
    except groq.APIStatusError as e:
        print(f"[{datetime.now()}] Groq returned an error (status {e.status_code}): {e}")
        return

    result = {"summary": summary_text, "last_updated": datetime.now().isoformat()}
    with open(SCOREBOARD_DIR / "ai_summary.json", "w") as f:
        json.dump(result, f)
    print(f"[{datetime.now()}] ai_summary.json updated:")
    print(" ", summary_text)

In [ ]:
import time

def run(interval_seconds=90, run_seconds=None):
    start = time.time()
    try:
        while run_seconds is None or (time.time() - start) < run_seconds:
            generate_and_write_summary()
            time.sleep(interval_seconds)
    except KeyboardInterrupt:
        pass
    print("Stopped.")

run()

[2026-07-28 13:10:02.679182] ai_summary.json updated:
  Top products like Similique LED Desk Lamp and Aliquam Bluetooth Speaker drive revenue. Completed orders account for a significant portion of revenue. Notably, orders are being fulfilled quickly, with an average time of 11.7 minutes. This operational efficiency supports the overall revenue growth.
[2026-07-28 13:11:34.557042] ai_summary.json updated:
  Revenue is driven by top-selling products like Similique LED Desk Lamp and Aliquam Bluetooth Speaker. Completed orders account for a significant portion of revenue. Notably, orders are being fulfilled quickly, with an average time of 11.6 minutes, indicating efficient operations.
[2026-07-28 13:13:05.627268] ai_summary.json updated:
  Top products like Similique LED Desk Lamp are driving revenue. Most orders are in processing or pending status. The average fulfillment time is 11.4 minutes, indicating efficient operations. Revenue is spread across various states, with Chhattisgarh lea